In [1]:
import numpy as np

# Dynamical Critical Exponent from Nonlinear Fluctuating Hydrodynamics

The goal of the rates matrix classification is to predict the universality class of each hydrodynamic mode before running simulations. For a multi species exclusion process, the rates matrix tells us how different species exchange positions. From this matrix, we can calculate the hydrodynamic currents, the normal modes, and the nonlinear mode-coupling coefficients. These coefficients tell us whether each mode should be KPZ or diffusive. This section follows [Kaufmann and Schütz](https://www.math.purdue.edu/~ebkaufma/PhysRevE.96.032119.pdf).

## Rates Matrix

For a three-species system, the rates matrix has the form

$$
g =
\begin{pmatrix}
0 & g_{01} & g_{02} \\
g_{10} & 0 & g_{12} \\
g_{20} & g_{21} & 0
\end{pmatrix}
$$

Here, $g_{ab}$ is the rate at which species $a$ swaps with species $b$ in a chosen direction. The diagonal entries are zero because a particle does not exchange with another particle of the same species.

The important part is not just the rate $g_{ab}$, but the asymmetry between $g_{ab}$ and $g_{ba}$. We define

$$
A_{ab} = g_{ab} - g_{ba}
$$

This antisymmetric part controls the macroscopic current. If $A_{ab} = 0$, then species $a$ and $b$ exchange symmetrically and there is no preferred drift between those two species.

In [2]:
"""
The asymmetric part of the exchange rates.
"""
def antisymmetric_rate_matrix(rates_matrix):
    rates_matrix = np.asarray(rates_matrix, dtype=float)
    return rates_matrix - rates_matrix.T

## Hydrodynamic Currents

The system has conserved densities. For three species, the densities are

$$
\rho_0,\rho_1,\rho_2
$$

but they satisfy

$$
\rho_0 + \rho_1 + \rho_2 = 1
$$

so only two of them are independent. Usually we use species $0$ as the reference species and treat $\rho_1$ and $\rho_2$ as the independent fields.

The hydrodynamic current for species $a$ is

$$
j_a(\rho) = \rho_a \sum_b A_{ab}\rho_b
$$

This says that the current of species $a$ depends on how much of species $a$ is present and on its asymmetric exchange with all other species.

To understand small fluctuations around the average density, we linearize the currents. This gives the current Jacobian

$$
J_{ab} = \frac{\partial j_a}{\partial \rho_b}
$$

The Jacobian tells us how small density perturbations move through the system. Its eigenvalues are the mode velocities, and its eigenvectors define the hydrodynamic normal modes.

For a three-species system, there are two independent conserved fields, so there are two hydrodynamic modes.

In [3]:
"""
Compute the hydrodynamic currents for the independent species.
"""
def current_vector(rates_matrix, density):
    rates_matrix = np.asarray(rates_matrix, dtype=float)
    density = np.asarray(density, dtype=float)

    dimension = rates_matrix.shape[0]
    n_modes = dimension - 1

    A = antisymmetric_rate_matrix(rates_matrix)

    j = np.zeros(n_modes, dtype=float)

    for a_ind in range(n_modes):
        a = a_ind + 1

        total = 0.0

        for b in range(dimension):
            total += A[a, b] * density[b]

        j[a_ind] = density[a] * total

    return j

In [4]:
"""
Compute the Jacobian of the independent currents with respect to the independent densities
"""
def current_jacobian(rates_matrix, density):
    rates_matrix = np.asarray(rates_matrix, dtype=float)
    density = np.asarray(density, dtype=float)

    dimension = rates_matrix.shape[0]
    n_modes = dimension - 1

    A = antisymmetric_rate_matrix(rates_matrix)

    J = np.zeros((n_modes, n_modes), dtype=float)

    for a_ind in range(n_modes):
        a = a_ind + 1
        rho_a = density[a]

        A_a0 = A[a, 0]

        bracket = A_a0

        for c_ind in range(n_modes):
            c = c_ind + 1
            bracket += (A[a, c] - A_a0) * density[c]

        for b_ind in range(n_modes):
            b = b_ind + 1

            diagonal_piece = bracket if a_ind == b_ind else 0.0
            derivative_piece = rho_a * (A[a, b] - A_a0)

            J[a_ind, b_ind] = diagonal_piece + derivative_piece

    return J

## Hydrodynamic Normal Modes

The raw density fields are usually coupled to each other. That means a fluctuation in one species density can affect the other species density. To separate the system into independent moving modes, we diagonalize the current Jacobian.

This gives a transformation matrix $R$ that converts the original density fields into normal modes.

In simple terms:

$$
\text{density fields} \rightarrow \text{normal modes}
$$

The normal modes are the natural large-scale variables of the system. These are the fields that should show KPZ or diffusive scaling.

The linear theory only tells us how fast the modes move. It does not tell us whether a mode is KPZ or diffusive. To classify the modes, we need the quadratic nonlinear terms in the current expansion.

We expand the current around the average density:

$$
j(\rho + u) = j(\rho) + J u + \frac{1}{2} H(u,u) + \cdots
$$

Here, $H$ is the Hessian of the current. It contains the second derivatives

$$
H^a_{bc} = \frac{\partial^2 j_a}{\partial \rho_b \partial \rho_c}
$$

These second derivatives measure the nonlinear coupling between density fluctuations.


In [5]:
"""
Compute Hessians of the independent currents.
"""

def current_hessians(rates_matrix, density):
    rates_matrix = np.asarray(rates_matrix, dtype=float)
    density = np.asarray(density, dtype=float)

    dimension = rates_matrix.shape[0]
    n_modes = dimension - 1

    A = antisymmetric_rate_matrix(rates_matrix)

    H = np.zeros((n_modes, n_modes, n_modes), dtype=float)

    for lam in range(n_modes):
        a = lam + 1
        A_a0 = A[a, 0]

        for b_ind in range(n_modes):
            b = b_ind + 1

            for c_ind in range(n_modes):
                c = c_ind + 1

                value = 0.0

                if lam == b_ind:
                    value += A[a, c] - A_a0

                if lam == c_ind:
                    value += A[a, b] - A_a0

                H[lam, b_ind, c_ind] = value

    return H

## Mode-Coupling Tensor

The Hessian is first written in the original density basis. But the universality class is determined in the normal-mode basis. So we rotate the Hessian into the normal-mode coordinates.

This gives the mode-coupling tensor

$$
G^\gamma_{\alpha\beta}
$$

This number tells us how modes $\alpha$ and $\beta$ couple into mode $\gamma$.

The most important entries are the self-couplings

$$
G^\gamma_{\gamma\gamma}
$$

These measure whether mode $\gamma$ couples nonlinearly to itself.

The classification rule is simple:

If the self-coupling of a mode is nonzero, then that mode is KPZ:

$$
G^\gamma_{\gamma\gamma} \neq 0
$$

so

$$
z_\gamma = 3/2
$$

If the self-coupling is zero, then that mode is diffusive, assuming there is no other relevant nonlinear coupling forcing a different class:

$$
G^\gamma_{\gamma\gamma} = 0
$$

so

$$
z_\gamma = 2
$$

So the classification comes from the self-coupling coefficients, not from fitting simulation data.


In [6]:
"""
Compute the left eigenvector matrix R.
"""
def left_eigenvector_matrix(J, tol=1e-10):
    J = np.asarray(J, dtype=float)

    eigenvalues, eigenvectors = np.linalg.eig(J.T)

    if np.max(np.abs(eigenvalues.imag)) > tol:
        raise ValueError("Current Jacobian has complex eigenvalues.")

    if np.max(np.abs(eigenvectors.imag)) > tol:
        raise ValueError("Left eigenvectors have non-negligible imaginary parts.")

    eigenvalues = eigenvalues.real
    eigenvectors = eigenvectors.real

    order = np.argsort(eigenvalues)

    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    n_modes = J.shape[0]
    R = np.zeros((n_modes, n_modes), dtype=float)

    for gamma in range(n_modes):
        v = eigenvectors[:, gamma]

        norm = np.linalg.norm(v)

        if norm < 1e-14:
            raise ValueError("Eigenvector has near-zero norm.")

        R[gamma, :] = v / norm

    return eigenvalues, R


In [7]:
"""
Compute the nonlinear mode-coupling tensor G.
"""
def mode_coupling_tensor(R, H):
    R = np.asarray(R, dtype=float)
    H = np.asarray(H, dtype=float)

    n_modes = R.shape[0]
    R_inv = np.linalg.inv(R)

    G = np.zeros((n_modes, n_modes, n_modes), dtype=float)

    for gamma in range(n_modes):
        for alpha in range(n_modes):
            for beta in range(n_modes):

                value = 0.0

                for lam in range(n_modes):
                    rotated_H_lam = R_inv.T @ H[lam] @ R_inv
                    value += R[gamma, lam] * rotated_H_lam[alpha, beta]

                G[gamma, alpha, beta] = 0.5 * value

    return G


In [8]:
"""
Classify each mode using its self-coupling.
"""
def classify_mode_couplings(G, tol=1e-10):
    G = np.asarray(G, dtype=float)

    n_modes = G.shape[0]
    classifications = []

    for gamma in range(n_modes):
        self_coupling = G[gamma, gamma, gamma]

        if abs(self_coupling) > tol:
            classifications.append("KPZ")
        else:
            classifications.append("diffusive")

    return classifications

In [17]:
"""
Pipeline: rates matrix -> current Jacobian -> left eigenvectors / normal modes -> current Hessians -> mode-coupling tensor -> classification
"""
def classify_rates_matrix(rates_matrix, density=None, tol=1e-10):
    rates_matrix = np.asarray(rates_matrix, dtype=float)

    dimension = rates_matrix.shape[0]

    if density is None:
        density = np.ones(dimension) / dimension
    else:
        density = np.asarray(density, dtype=float)

    J = current_jacobian(rates_matrix, density)
    H = current_hessians(rates_matrix, density)

    eigenvalues, R = left_eigenvector_matrix(J, tol=tol)

    G = mode_coupling_tensor(R, H)

    classifications = classify_mode_couplings(G, tol=tol)


    for gamma, label in enumerate(classifications):
        print(f"mode {gamma}: ,G[{gamma},{gamma},{gamma}] = {G[gamma, gamma, gamma]:.12f}, classification = {label}")

In [18]:
rates_matrix = np.array([[0.0, 1.0, 0.1], [1.0, 0.0, 0.1], [2.1, 2.1, 0.0]])
classify_rates_matrix(rates_matrix)

mode 0: ,G[0,0,0] = 0.000000000000, classification = diffusive
mode 1: ,G[1,1,1] = -2.000000000000, classification = KPZ


In [19]:
rates_matrix = np.array([[0.0, 1.0, 1.0], [0.0, 0.0, 1.0], [0.0, 0.0, 0.0]])
classify_rates_matrix(rates_matrix)

mode 0: ,G[0,0,0] = 1.000000000000, classification = KPZ
mode 1: ,G[1,1,1] = 1.414213562373, classification = KPZ


In [20]:
rates_matrix = np.array([[0.0, 1.0, 1.0], [1.0, 0.0, 1.0], [1.0, 1.0, 0.0]])
classify_rates_matrix(rates_matrix)

mode 0: ,G[0,0,0] = 0.000000000000, classification = diffusive
mode 1: ,G[1,1,1] = 0.000000000000, classification = diffusive
